# T30 / E13 — Đổi mô hình đọc sang Sailor2-8B

**Câu hỏi của thí nghiệm này không phải "mô hình nào điểm cao hơn".**

Sailor2-8B-SFT mở rộng từ chính họ Qwen2.5 rồi huấn luyện thêm rất nhiều trên ngôn ngữ Đông Nam
Á. Nên nó là một phép thử tự nhiên: **vị trí các đầu chú ý có ích là thuộc tính của kiến trúc,
hay của dữ liệu huấn luyện?**

Nếu các đầu mà bộ dò tuyến tính dựa vào nằm ở cùng độ sâu trong cả hai mô hình, thì vị trí đầu
sao chép sống sót qua việc huấn luyện chuyên sâu một ngôn ngữ khác. Nếu chúng dịch chuyển thì
ngược lại. **Cả hai đều là kết quả** — kết quả thứ hai đừng lược đi.

Bài gốc Lookback Lens có một kết quả liên quan: bộ dò huấn luyện trên mô hình 7B dùng lại được
cho 13B mà không cần huấn luyện lại. Ô này kiểm điều tương tự theo hướng khác — không đổi cỡ mà
đổi dữ liệu huấn luyện.

## Khác T27 ở ba chỗ, và cả ba đều là chỗ dễ hỏng

1. **Lớp tràn số của Sailor2 chưa biết.** Với Qwen2.5-7B, T07 đo được đúng lớp 27 — lớp cuối —
   hỏng ở `float16` trên 20/20 mẫu. Sailor2 là mô hình khác, nhiều lớp hơn, nên lớp hỏng của nó
   là **câu hỏi thực nghiệm**. Ô 5 đo trước rồi tự ghi vào cấu hình; `exclude_layers` để trống
   trong repo là cố ý.
2. **Lưới lớp × đầu có thể khác.** Qwen cho 28 lớp; Sailor2 là bản mở rộng nên nhiều hơn. Phần
   so vị trí đầu xử lý riêng trường hợp này, xem ô 9.
3. **Mẫu prompt.** Chat template của Sailor2 có thể khác Qwen đôi chút. Không phải lo: từ T07,
   vị trí ngữ cảnh và phản hồi được tìm bằng cách **dò chuỗi trong prompt đã render** rồi ánh xạ
   sang token, chứ không đếm ký tự khung. Ô 6 in ra để đối chiếu.

## Bộ nhớ — chỗ lượt chạy đầu chết

Sailor2-8B **không nạp nổi ở `float32`** trên T4: lượt chạy 06/09 hết bộ nhớ ở ô 5, tại 14,4/14,56
GiB, ngay khi nạp trọng số. Mô hình lớn hơn Qwen2.5-7B và mang từ điển Đông Nam Á rộng hơn, mà bộ
nạp thì chuyển từng tensor sang `float32` trên GPU trước khi lượng tử hóa.

Ô 5 nay dùng `--reference bfloat16` và chạy lượt `float16` trước. Chi tiết ở phần trước ô 5.

Lượt trích chính (ô 7) chạy ở `float16` + NF4 nên không dính vấn đề này.

## Chi phí

Khoảng **3 giờ card đồ họa** cho 7.000 mẫu ViHallu, cộng ~10 phút cho ô dò kiểu số. Nằm gọn
trong hạn mức 30 giờ/tuần.

**Chấm điểm chạy ở máy cá nhân**, theo quy tắc chốt ở T23: điểm dev lệch tới 0,0075 giữa hai môi
trường vì bộ giải tối ưu hội tụ khác nhau. Notebook này chỉ trích đặc trưng.


## Chuẩn bị

Ô 3 là ô tiền kiểm. Ô 4 chuẩn hóa và chia tập — **đừng bỏ**, đây đúng chỗ lượt chạy T27 hỏng
lần thứ hai.

Ô 4 chạy `tests/test_attention_hook.py` **trên chính Kaggle**. Notebook cài bằng `--no-deps` nên
dùng bản `transformers` của Kaggle chứ không phải bản ghim trong `pyproject.toml` — và mục 5
`CLAUDE.md` cảnh báo việc hook có nhận được `attn_weights` hay không **phụ thuộc phiên bản**. Bộ
kiểm thử đó vốn chỉ chạy ở máy cá nhân, tức đúng môi trường **không** rủi ro. Nó tốn vài giây và
chạy trước khi đụng tới GPU.


In [1]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    print("$", " ".join(str(a) for a in args))
    result = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    print(result.stdout.strip())
    if result.returncode:
        print(result.stderr.strip())
        raise SystemExit(f"lệnh hỏng: {' '.join(str(a) for a in args)}")
    return result.stdout


if REPO_DIR.exists():
    run("git", "fetch", "--all", cwd=REPO_DIR)
    run("git", "reset", "--hard", "origin/main", cwd=REPO_DIR)
else:
    run("git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR))

os.chdir(REPO_DIR)
run("git", "log", "-1", "--format=%h %s")

$ git clone --depth 1 https://github.com/wsunicorn/vihallulens.git /kaggle/working/vihallulens

$ git log -1 --format=%h %s
5f3a4c4 T30: ô kiểm chặn luôn đường lấy shard về, thêm công cụ soi shard (#82)


'5f3a4c4 T30: ô kiểm chặn luôn đường lấy shard về, thêm công cụ soi shard (#82)\n'

In [2]:
# Ô 2 — cài đặt. bitsandbytes cần cho lượng tử hóa 4 bit.
#
# Sailor2-8B nặng hơn Qwen2.5-7B nên phần tải trọng số lâu hơn, khoảng 8-10 phút lần đầu.
!pip install -q --no-deps -e .
!pip install -q bitsandbytes

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vihallulens (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vihallulens 0.1.0 requires pyvi, which is not installed.
vihallulens 0.1.0 requires rank-bm25, which is not installed.


In [3]:
# Ô 3 — TIỀN KIỂM. Vài giây, chạy trước mọi thứ.
#
# Cùng lối viết đã chốt ở T27: đi HẾT chuỗi phụ thuộc, tách rành mạch hai nhóm.
#
#   PHẢI CÓ SẴN  - phiên này không tạo được: dữ liệu thô đã mount, gói đã cài
#   TỰ TẠO       - các ô sau sinh ra theo thứ tự, liệt kê để đọc và đối chiếu
import importlib.util
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import extraction_hash, load_config
from vihallulens.data.paths import find_raw_dir

CFG = "configs/e13_sailor2_vihallu.yaml"
cfg = load_config(CFG)
problems = []

# --- PHẢI CÓ SẴN -------------------------------------------------------------------------------
packages = ("torch", "transformers", "bitsandbytes", "pandas", "accelerate")
absent = [name for name in packages if importlib.util.find_spec(name) is None]
trang_thai = f"THIEU {absent}" if absent else f"du ca {list(packages)}"
print(f"  goi phai co san   : {trang_thai}")
if absent:
    problems.append(f"thieu goi {absent}")

try:
    raw = find_raw_dir()
    files = sorted(p.name for p in Path(raw).glob("vihallu*"))
    print(f"  du lieu tho       : {raw}")
    print(f"  file vihallu tho  : {files or 'KHONG CO'}")
    if not files:
        problems.append("khong thay file vihallu nao trong du lieu tho")
except Exception as error:
    print(f"  du lieu tho       : KHONG TIM THAY ({error})")
    problems.append("chua mount dataset du lieu tho")

print(f"  mo hinh doc       : {cfg.extractor.model_name}")
print(f"  exclude_layers    : {cfg.extractor.exclude_layers}  <- o 5 se ghi de")

# --- TỰ TẠO, theo thứ tự các ô sau ---------------------------------------------------------------
chain = [
    ("o 4", "data/interim/vihallu_{train,dev,test}.parquet", "normalize_data + split_data"),
    ("o 5", "configs/e13_sailor2_vihallu.yaml co exclude_layers", "compare_dtypes"),
    ("o 7", "data/processed/vihallu_{split}_<hash>.jsonl", "extract_features"),
]
print("  chuoi tu tao:")
for cell, target, maker in chain:
    print(f"    {cell:<5} {target:<52} <- {maker}")

if problems:
    raise SystemExit("TIEN KIEM HONG: " + "; ".join(problems))
print("\nTien kiem dat: dieu kien ngoai da du, phan con lai phien nay tu tao.")

  goi phai co san   : du ca ['torch', 'transformers', 'bitsandbytes', 'pandas', 'accelerate']
  du lieu tho       : /kaggle/input/datasets/unicorn1209/vihallulens
  file vihallu tho  : ['vihallu_test_public.csv', 'vihallu_train.csv']
  mo hinh doc       : sail/Sailor2-8B-SFT
  exclude_layers    : []  <- o 5 se ghi de
  chuoi tu tao:
    o 4   data/interim/vihallu_{train,dev,test}.parquet        <- normalize_data + split_data
    o 5   configs/e13_sailor2_vihallu.yaml co exclude_layers   <- compare_dtypes
    o 7   data/processed/vihallu_{split}_<hash>.jsonl          <- extract_features

Tien kiem dat: dieu kien ngoai da du, phan con lai phien nay tu tao.


In [4]:
# Ô 4 — chuẩn bị dữ liệu và kiểm môi trường. Khoảng 2 phút, CPU.
#
# tests/test_attention_hook.py chay o day chu khong chi o may ca nhan, va day la chỗ no can
# chay nhat: notebook cai bang --no-deps nen dung ban transformers cua Kaggle, khong phai ban
# ghim trong pyproject.toml. Muc 5 CLAUDE.md canh bao viec hook co nhan duoc attn_weights hay
# khong PHU THUOC PHIEN BAN — nen phai hoi chinh moi truong sap tieu 3 gio GPU, trong vai giay,
# tren mo hinh Qwen2 hai lop trong so ngau nhien chay CPU.
!python scripts/probe_env.py
!python scripts/normalize_data.py --dataset vihallu
!python scripts/split_data.py --only vihallu
!python -m pytest tests/test_attention_hook.py tests/test_compare_heads.py -q


MÔI TRƯỜNG
  repo             : /kaggle/working/vihallulens
  commit           : 5f3a4c4 T30: ô kiểm chặn luôn đường lấy shard về, thêm công cụ soi shard (#82)
  python           : 3.12.13
  torch            : 2.10.0+cu128
  transformers     : 5.0.0
  bitsandbytes     : 0.50.2
  accelerate       : 1.13.0
  vihallulens      : 0.1.0 tại /kaggle/working/vihallulens/src/vihallulens/__init__.py
  dữ liệu          : /kaggle/input/datasets/unicorn1209/vihallulens  (14 file)
      MANIFEST.md
      isedsc01_test_private.json
      isedsc01_test_public.json
      isedsc01_train.json
      vifactcheck_dataset_card.md
      vifactcheck_dev.parquet
      vifactcheck_gitattributes.txt
      vifactcheck_test.parquet
      vifactcheck_train.parquet
      vihallu_test_public.csv
      vihallu_train.csv
      viwikifc_dev.csv
      viwikifc_test.csv
      viwikifc_train.csv

CHUẨN HÓA VIHALLU
  nguồn                 : /kaggle/input/datasets/unicorn1209/vihallulens
  số dòng               : 7,000
  ngữ

## Dò lớp tràn số — bắt buộc, và không suy ra được từ Qwen

Ô 5 tốn khoảng 10 phút GPU và **quyết định cả lượt chạy 3 giờ phía sau**.

**Lượt chạy đầu ngày 06/09 chết ở đây**, hết bộ nhớ khi nạp Sailor2 ở `float32`. Nguyên nhân: bộ
nạp của `transformers` chuyển từng tensor sang `float32` trên GPU **trước khi** bitsandbytes
lượng tử hóa, còn embedding và lm_head thì ở nguyên `float32` sau đó. Sailor2 lớn hơn Qwen2.5-7B
và mang từ điển Đông Nam Á rộng hơn, nên không còn vừa 14,56 GiB.

Ba thứ đã sửa:

1. **Lượt `float16` chạy trước.** Danh sách lớp tràn số chỉ cần lượt này — lượt mốc trả lời câu
   hỏi khác, là các lớp còn sống có bị bóp méo không. Thứ tự cũ vứt mất câu trả lời đã nằm trong
   tầm tay khi nửa tùy chọn hết bộ nhớ.
2. **Lượt mốc được phép hỏng.** Hết bộ nhớ ở đó thì in cảnh báo rồi bỏ bảng so lệch, không giết
   cả script.
3. **Dùng `--reference bfloat16`.** `bfloat16` có **cùng dải mũ với `float32`** nên không tràn ở
   chỗ `float16` tràn, mà tốn bộ nhớ ngang `float16`. Với mô hình không vừa mốc `float32` thì nó
   còn là mốc hợp lý hơn: cả Qwen2.5 lẫn Sailor2 đều được huấn luyện ở `bfloat16`.

**Đọc gì:** danh sách lớp tràn số. Với Qwen là đúng một lớp, lớp cuối. Nếu Sailor2 ra nhiều lớp
hoặc ra lớp ở giữa thì dừng lại đọc bảng per-layer — mất nhiều lớp giữa sẽ làm phép so với Qwen
khập khiễng và phải ghi rõ trong báo cáo.

Ô 5 tự ghi kết quả vào `configs/e13_sailor2_vihallu.yaml`. **Nhớ commit lại file đó** sau khi
chạy xong, nếu không lượt chạy này không tái lập được.

In [5]:
# Ô 5 — DÒ LỚP TRÀN SỐ CỦA SAILOR2. Khoảng 10 phút GPU. BẮT BUỘC chạy trước ô 7.
#
# Đây là ô quan trọng nhất notebook này. Với Qwen2.5-7B, T07 đo được đúng lớp cuối tràn số ở
# float16 trên 20/20 mẫu, và bỏ nó đi thì 27 lớp còn lại khớp float32 tới 0,07 % thang đo.
#
# Sailor2 là mô hình KHÁC. Lớp hỏng của nó không suy ra được từ Qwen — phải đo. Bỏ qua ô này thì
# mọi đặc trưng trích ra sẽ có nan ở ít nhất một lớp, và điều đó chỉ lộ ra sau 3 giờ GPU.
#
# --reference bfloat16 chứ không phải float32: lượt chạy 06/09 hết bộ nhớ khi nạp Sailor2 ở
# float32. bfloat16 có cùng dải mũ với float32 nên không tràn ở chỗ float16 tràn, mà tốn bộ nhớ
# ngang float16.
import ast
import os
import re
from pathlib import Path

# Giảm phân mảnh bộ nhớ, đúng thứ thông báo lỗi lần trước gợi ý.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

proc = subprocess.run(
    ["python", "scripts/compare_dtypes.py", "--model", "sail/Sailor2-8B-SFT",
     "--per-dataset", "10", "--reference", "bfloat16"],
    capture_output=True, text=True, env={**os.environ},
)
print(proc.stdout[-8000:])
if proc.returncode:
    print(proc.stderr[-4000:])
    raise SystemExit("do kieu so hong")

# compare_dtypes.py in ra mot dong may doc duoc: EXCLUDE_LAYERS=[27]
found = re.search(r"^EXCLUDE_LAYERS=(\[.*\])$", proc.stdout, flags=re.MULTILINE)
if not found:
    raise SystemExit("khong thay dong EXCLUDE_LAYERS= trong output — doc ky stdout ben tren")
bad = sorted(set(ast.literal_eval(found.group(1))))
print()
print(f"  Lop tran so do duoc: {bad if bad else 'KHONG CO'}")

if not bad:
    raise SystemExit(
        "Khong lop nao tran so. Voi Qwen thi lop cuoi luon tran, nen ket qua nay dang ngo.\n"
        "Hai kha nang, va phai phan biet duoc truoc khi chay tiep:\n"
        "  - Sailor2 that su on o float16. Co the that, va la mot phat hien dang ghi.\n"
        "  - Phep do chua cham toi lop hong vi 10 mau qua it hoac ngu canh qua ngan.\n"
        "Chay lai voi --per-dataset 20 roi doc bang per-layer ben tren truoc khi quyet."
    )

# Ghi thang vao cau hinh de extraction_hash phan anh dung thu da chay. Ghi de an toan vi
# exclude_layers trong repo de trong co chu dich.
CFG_PATH = Path("configs/e13_sailor2_vihallu.yaml")
text = CFG_PATH.read_text(encoding="utf-8")
patched = re.sub(r"^  exclude_layers: \[\]$", f"  exclude_layers: {bad}", text, count=1,
                 flags=re.MULTILINE)
if patched == text:
    raise SystemExit("khong tim thay dong 'exclude_layers: []' de ghi de — kiem tra lai config")
CFG_PATH.write_text(patched, encoding="utf-8")
print(f"  Da ghi vao config : exclude_layers: {bad}")
print("  NHO commit lai config nay sau khi chay xong, de luot chay tai lap duoc.")


SO SÁNH float16 VỚI BFLOAT16
  dữ liệu   : /kaggle/input/datasets/unicorn1209/vihallulens
  số mẫu    : 20
  độ dài    : 47 đến 4805 từ
    [float16] 1/20
    [float16] 2/20
    [float16] 3/20
    [float16] 4/20
    [float16] 5/20
    [float16] 6/20
    [float16] 7/20
    [float16] 8/20
    [float16] 9/20
    [float16] 10/20
    [float16] 11/20
    [float16] 12/20
    [float16] 13/20
    [float16] 14/20
    [float16] 15/20
    [float16] 16/20
    [float16] 17/20
    [float16] 18/20
    [float16] 19/20
    [float16] 20/20
    [float16] xong 20 mẫu trong 22 s

  !! Hết bộ nhớ khi nạp mô hình ở bfloat16. Bỏ phần so lệch.
     Danh sách lớp tràn số bên dưới VẪN ĐÚNG — nó chỉ cần lượt float16.
     Mất phần này nghĩa là chưa kiểm được các lớp còn sống ở float16 có bị
     bóp méo không. Phải ghi vào phần hạn chế, đừng lặng lẽ bỏ qua.

  float16  :     1068 ms mỗi mẫu, VRAM đỉnh     8785 MB

  Mẫu có ít nhất một lớp nan ở fp16 : 20/20
  Tập hợp các lớp từng nan          : [30, 31]
EXCLUDE_L

In [6]:
# Ô 6 — cổng kiểm trước khi tiêu 3 giờ GPU. Vài giây, CPU.
import sys

sys.path.insert(0, "src")
from importlib import reload

from transformers import AutoTokenizer

import vihallulens.config as config_module
from vihallulens.extract.prompt import render_prompt

reload(config_module)
cfg = config_module.load_config("configs/e13_sailor2_vihallu.yaml")
run = config_module.extraction_hash(cfg)

print(f"  mo hinh doc       : {cfg.extractor.model_name}")
print(f"  exclude_layers    : {cfg.extractor.exclude_layers}")
print(f"  hash trich        : {run}")

if not cfg.extractor.exclude_layers:
    raise SystemExit("exclude_layers van trong — o 5 chua chay hoac chua ghi duoc. DUNG LAI.")

# Chat template cua Sailor2 co the khac Qwen. Tu T07, vi tri ngu canh va phan hoi duoc tim bang
# cach DO CHUOI trong prompt da render roi anh xa sang token, chu khong dem ky tu khung. O day
# kiem thang tinh chat do thay vi chi in ra nhin bang mat.
NGU_CANH = "Hà Nội là thủ đô của Việt Nam."
CAU_HOI = "Thủ đô Việt Nam là gì?"
PHAN_HOI = "Thủ đô là Hà Nội."

tok = AutoTokenizer.from_pretrained(cfg.extractor.model_name)
sample = render_prompt(tok, context=NGU_CANH, question=CAU_HOI, response=PHAN_HOI)

print()
print("  --- prompt da render ---")
print(sample.text)
print("  --- het ---")
print(f"  vung ngu canh doc lai : {sample.context!r}")
print(f"  vung phan hoi doc lai : {sample.response!r}")

assert sample.context == NGU_CANH, "vung ngu canh lech — DUNG LAI, dung trich dac trung"
assert sample.response == PHAN_HOI, "vung phan hoi lech — DUNG LAI, dung trich dac trung"
print()
print("  Hai vung khop chinh xac, nen chat template khac Qwen cung khong lam lech vi tri.")

  mo hinh doc       : sail/Sailor2-8B-SFT
  exclude_layers    : [30, 31]
  hash trich        : f58b3bee5934

  --- prompt da render ---
<|im_start|>system
Bạn là trợ lý trả lời câu hỏi dựa trên ngữ cảnh được cung cấp.<|im_end|>
<|im_start|>user
Ngữ cảnh:
Hà Nội là thủ đô của Việt Nam.

Câu hỏi: Thủ đô Việt Nam là gì?<|im_end|>
<|im_start|>assistant
Thủ đô là Hà Nội.<|im_end|>

  --- het ---
  vung ngu canh doc lai : 'Hà Nội là thủ đô của Việt Nam.'
  vung phan hoi doc lai : 'Thủ đô là Hà Nội.'

  Hai vung khop chinh xac, nen chat template khac Qwen cung khong lam lech vi tri.


## Trích đặc trưng

Khoảng **3 giờ**. Chạy lại được — phần đã xong không mất.

**Đọc gì trong lúc chạy:** dòng `lớp bỏ` phải khớp danh sách ô 5 đo được, và `lỗi` phải là 0.
Nếu thấy `nan` xuất hiện thì dừng ngay: nghĩa là còn lớp tràn số mà ô 5 chưa bắt được.

In [7]:
# Ô 7 — trích đặc trưng. Khoảng 3 giờ. Chạy lại được, có lưu tiến độ.
#
# Ba tập chạy nối nhau. Nếu phiên đứt thì chạy lại ô này, phần đã xong không mất.
import os

# Tắt thanh tiến trình nạp trọng số. Lượt Sailor2 06/09 in ra hàng nghìn dòng
# "Loading weights: 26%| | 100/387 ... Materializing param=..." — mỗi lần vẽ lại một dòng, nhân
# ba lượt train/dev/test — và đẩy log vượt giới hạn 50.000 ký tự của trình xem, cắt mất đúng
# phần cần đọc: số mẫu xong, số lỗi, tỷ lệ cắt ngữ cảnh, VRAM đỉnh.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

!python scripts/extract_features.py --config configs/e13_sailor2_vihallu.yaml --split train
!python scripts/extract_features.py --config configs/e13_sailor2_vihallu.yaml --split dev
!python scripts/extract_features.py --config configs/e13_sailor2_vihallu.yaml --split test


T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e13_sailor2_vihallu.yaml  (hash f58b3bee5934)
  mô hình đọc           : sail/Sailor2-8B-SFT
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : [30, 31]
  trần token ngữ cảnh   : 4,096
  chia đoạn             : sentence
  bộ dữ liệu            : vihallu/train, 5,600 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  mẫu có bằng chứng     : 0/5,600
  đã có sẵn             : 0 mẫu trong vihallu_train_f58b3bee5934.jsonl
  còn phải chạy         : 5,600 mẫu
/kaggle/working/vihallulens/src/vihallulens/features/lookback.py:48: RuntimeWarning: Mean of empty slice
  pooled = np.nanmean(lookback.astype(np.float64), axis=2)


--------------------------------------------------------------------------------
  đã ghi thêm           : 5,600 mẫu, tổng 5,600
  lỗi                   : 0
  bị cắt ngữ cảnh       : 0/5,600
  có lớp tràn số    

In [8]:
# Ô 8 — soi shard trước khi rời phiên. Vài giây, CPU. KHÔNG dừng notebook.
#
# Ô này từng raise SystemExit khi shard có nan, và điều đó chặn luôn ô 9 — tức chặn đúng đường
# mang shard về để chẩn đoán. Một shard hỏng là thứ CẦN đem về nhất. Giờ nó chỉ báo động.
!python scripts/inspect_shard.py --config configs/e13_sailor2_vihallu.yaml


SOI SHARD — e13_sailor2_vihallu
  mô hình đọc       : sail/Sailor2-8B-SFT
  đang bỏ lớp       : [30, 31]
  hash trích        : f58b3bee5934

  TRAIN  5,600 mẫu   lưới 30 × 28
    mẫu có lớp tràn số : 38  (0.68 %)
    theo lớp           : lớp 0: 6, lớp 1: 38, lớp 2: 38, lớp 3: 38, lớp 4: 38, lớp 5: 38, lớp 6: 38, lớp 7: 38, lớp 8: 38, lớp 9: 38, lớp 10: 38, lớp 11: 38, lớp 12: 38, lớp 13: 38, lớp 14: 38, lớp 15: 38, lớp 16: 38, lớp 17: 38, lớp 18: 38, lớp 19: 38, lớp 20: 38, lớp 21: 38, lớp 22: 38, lớp 23: 38, lớp 24: 38, lớp 25: 38, lớp 26: 38, lớp 27: 38, lớp 28: 38, lớp 29: 38
      lookback_total    : 30,856 giá trị không hữu hạn trên 38 dòng
      lookback_context  : 30,856 giá trị không hữu hạn trên 38 dòng

  DEV  700 mẫu   lưới 30 × 28
    mẫu có lớp tràn số : 8  (1.14 %)
    theo lớp           : lớp 0: 1, lớp 1: 8, lớp 2: 8, lớp 3: 8, lớp 4: 8, lớp 5: 8, lớp 6: 8, lớp 7: 8, lớp 8: 8, lớp 9: 8, lớp 10: 8, lớp 11: 8, lớp 12: 8, lớp 13: 8, lớp 14: 8, lớp 15: 8, lớp 16: 8, lớp 17:

In [9]:
# Ô 8b — đọc kết luận rồi quyết. Vài giây, CPU.
import sys

sys.path.insert(0, "src")
sys.path.insert(0, "scripts")
from pathlib import Path

import numpy as np

from extract_features import load_done, shard_path
from vihallulens.config import load_config

cfg = load_config("configs/e13_sailor2_vihallu.yaml")
run = extraction_hash(cfg)

grid = None
for split in ("train", "dev", "test"):
    rows = list(load_done(shard_path(Path("data/processed"), run, "vihallu", split)).values())
    if not rows:
        print(f"  {split:<6}: TRONG")
        continue
    n_layers = len(rows[0]["layer_indices"])
    n_heads = len(rows[0]["lookback_total"]) // n_layers
    grid = (n_layers, n_heads)
    values = np.asarray([r["lookback_total"] for r in rows], dtype=np.float32)
    finite = np.isfinite(values)
    print(f"  {split:<6}: {len(rows):>6,} mau | luoi {n_layers} x {n_heads} "
          f"| dong co nan {int((~finite).any(axis=1).sum()):,}")

print()
print(f"  Luoi cua Sailor2: {grid[0]} lop x {grid[1]} dau = {grid[0] * grid[1]:,} cap")
print("  Qwen2.5-7B de so: 27 lop x 28 dau = 756 cap")
if grid != (27, 28):
    print("  -> HAI LUOI KHAC NHAU. Phan so vi tri dau se chi bao do sau tuong doi.")
print()
print("  DU SHARD CO NAN HAY KHONG, VAN CHAY O 9 DE MANG VE.")
print("  Bang chan doan o tren noi ro nen bo them lop (phai trich lai) hay bo may mau hong")
print("  (khong ton GPU). Quyet dinh do lam o may ca nhan, khong lam voi.")

  train :  5,600 mau | luoi 30 x 28 | dong co nan 38
  dev   :    700 mau | luoi 30 x 28 | dong co nan 8
  test  :    700 mau | luoi 30 x 28 | dong co nan 3

  Luoi cua Sailor2: 30 lop x 28 dau = 840 cap
  Qwen2.5-7B de so: 27 lop x 28 dau = 756 cap
  -> HAI LUOI KHAC NHAU. Phan so vi tri dau se chi bao do sau tuong doi.

  DU SHARD CO NAN HAY KHONG, VAN CHAY O 9 DE MANG VE.
  Bang chan doan o tren noi ro nen bo them lop (phai trich lai) hay bo may mau hong
  (khong ton GPU). Quyet dinh do lam o may ca nhan, khong lam voi.


## Chấm điểm — KHÔNG chạy ở đây

Theo quy tắc chốt ở T23, mọi phép so sánh phải chấm trên **cùng một máy**. Điểm dev lệch tới
0,0075 giữa Kaggle và máy cá nhân vì bộ giải tối ưu của hồi quy logistic hội tụ khác nhau giữa
hai phiên bản Python — mà biên độ đề tài đang xét chỉ cỡ 0,01.

Ô 9 gói shard lại để tải về. Chấm ở máy.

In [10]:
# Ô 9 — lấy kết quả về. Vài giây.
#
# Tải ba file này về máy cá nhân rồi chạy CHẤM ĐIỂM Ở ĐÓ, theo quy tắc chốt ở T23.
import shutil
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import extraction_hash, load_config

cfg = load_config("configs/e13_sailor2_vihallu.yaml")
run = extraction_hash(cfg)

out = Path("/kaggle/working/ket_qua_t30")
out.mkdir(exist_ok=True)
for split in ("train", "dev", "test"):
    src = Path(f"data/processed/vihallu_{split}_{run}.jsonl")
    if src.exists():
        shutil.copy(src, out / src.name)
shutil.copy("configs/e13_sailor2_vihallu.yaml", out / "e13_sailor2_vihallu.yaml")

for f in sorted(out.iterdir()):
    print(f"  {f.name:<44} {f.stat().st_size / 1e6:>8.1f} MB")

print("""
Tai het thu muc ket_qua_t30 ve may, dat vao:
  *.jsonl  ->  data/processed/
  e13_sailor2_vihallu.yaml  ->  configs/   (GHI DE — no mang exclude_layers da do)

Roi chay o may ca nhan:
  python scripts/run_chunk_aware.py --config configs/e13_sailor2_vihallu.yaml
  python scripts/compare_heads.py --config-a configs/e03_chunk_sentence_vihallu.yaml \\
                                  --config-b configs/e13_sailor2_vihallu.yaml
""")

  e13_sailor2_vihallu.yaml                          0.0 MB
  vihallu_dev_f58b3bee5934.jsonl                   40.7 MB
  vihallu_test_f58b3bee5934.jsonl                  40.9 MB
  vihallu_train_f58b3bee5934.jsonl                325.8 MB

Tai het thu muc ket_qua_t30 ve may, dat vao:
  *.jsonl  ->  data/processed/
  e13_sailor2_vihallu.yaml  ->  configs/   (GHI DE — no mang exclude_layers da do)

Roi chay o may ca nhan:
  python scripts/run_chunk_aware.py --config configs/e13_sailor2_vihallu.yaml
  python scripts/compare_heads.py --config-a configs/e03_chunk_sentence_vihallu.yaml \
                                  --config-b configs/e13_sailor2_vihallu.yaml

